Libraries

In [2]:
import os
import cv2
import numpy as np
import dtcwt
from tqdm import tqdm
from sklearn.preprocessing import normalize
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, StratifiedKFold

DTCWT Transformer

In [3]:
transform = dtcwt.Transform2d()

Application of DTCWT and DNA-Encoding

In [ ]:
def dtcwt_45_features(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray = gray.astype(np.float32) / 255.0
    coeffs = transform.forward(gray, nlevels=3)
    features = []

    for level in range(len(coeffs.highpasses)):
        highpass = coeffs.highpasses[level]
        subband_45 = highpass[:,:,1]
        magnitude = np.abs(subband_45)
        features.append(np.mean(magnitude))
        features.append(np.std(magnitude))
        features.append(np.energy(magnitude))

    return np.array(features)

def dna_features(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray = gray.astype(np.uint8)
    pixels = gray.flatten()
    dna_counts = {"A":0, "C":0, "G":0, "T":0}

    for p in pixels:
        b = format(p, '08b')

        for i in range(0,8,2):
            pair = b[i:i+2]
            if pair == "00":
                dna_counts["A"] += 1
            elif pair == "01":
                dna_counts["C"] += 1
            elif pair == "10":
                dna_counts["G"] += 1
            elif pair == "11":
                dna_counts["T"] += 1

    total = sum(dna_counts.values())

    return np.array([
        dna_counts["A"]/total,
        dna_counts["C"]/total,
        dna_counts["G"]/total,
        dna_counts["T"]/total
    ])


Loading CNN features

In [ ]:
f_cnn = np.load("/home/utkarshs/Desktop/Cbir_Video/Features")
labels = np.load("Features/labels.npy")